In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

ROOT = Path("/home/chenyh/workspace/fluProfiler")
RESULT_ROOT = ROOT / "results/H1H3_new"

# -----------------------------
# New data: SerumGate-subtype
# -----------------------------
def load_serumgate_subtype(split):
    paths = {
        "titer": [
            RESULT_ROOT / "SerumGate/titer/subtype/H1N1/refit_train_valid_e163_q32_s42_subtype0/predictions_test.csv",
            RESULT_ROOT / "SerumGate/titer/subtype/H3N2/refit_train_valid_e169_q32_s42_subtype0/predictions_test.csv",
        ],
        "strain": [
            RESULT_ROOT / "SerumGate/strain/subtype/H1N1/refit_train_valid_e123_q32_s42_subtype0/predictions_test.csv",
            RESULT_ROOT / "SerumGate/strain/subtype/H3N2/refit_train_valid_e132_q32_s42_subtype0/predictions_test.csv",
        ],
        "serum": [
            RESULT_ROOT / "SerumGate/serum/subtype/H1N1/refit_train_valid_e51_q32_s42_subtype0/predictions_test.csv",
            RESULT_ROOT / "SerumGate/serum/subtype/H3N2/refit_train_valid_e132_q32_s42_subtype0/predictions_test.csv",
        ],
    }[split]
    df = pd.concat([pd.read_csv(p) for p in paths], axis=0, ignore_index=True)
    return df.rename(columns={"mean": "prediction"})

# -----------------------------
# New data: Nextflu/AdaBoost without name
# -----------------------------
def load_nextflu_without_name(split):
    df = pd.read_csv(RESULT_ROOT / f"nextflu_merge_valid/{split}/predictions.csv")
    return df.rename(columns={"pred_without_name": "prediction"})

def load_adaboost_without_name(split):
    df = pd.read_csv(RESULT_ROOT / f"adaboost_pairwise_merge_valid/{split}/predictions.csv")
    if "split" in df.columns:
        df = df[df["split"].astype(str).str.lower() == "test"].copy()
    return df.rename(columns={"prediction_without_name": "prediction"})

serumgate_titer = load_serumgate_subtype("titer")
serumgate_strain = load_serumgate_subtype("strain")
serumgate_serum = load_serumgate_subtype("serum")

nextflu_titer = load_nextflu_without_name("titer")
nextflu_strain = load_nextflu_without_name("strain")
nextflu_serum = load_nextflu_without_name("serum")

adaboost_titer = load_adaboost_without_name("titer")
adaboost_strain = load_adaboost_without_name("strain")
adaboost_serum = load_adaboost_without_name("serum")

# -----------------------------
# VaxSeer unchanged: keep original values
# -----------------------------
if "vaxseer_titer" not in globals():
    vaxseer_titer_H1N1 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/titer_holdout/h1n1/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_titer_H3N2 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/titer_holdout/h3n2/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_titer = pd.concat([vaxseer_titer_H1N1, vaxseer_titer_H3N2], axis=0)
    vaxseer_titer.columns = ["src_id1", "src_id2", "prediction", "label"]

    vaxseer_strain_H1N1 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/strain_holdout/h1n1/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_strain_H3N2 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/strain_holdout/h3n2/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_strain = pd.concat([vaxseer_strain_H1N1, vaxseer_strain_H3N2], axis=0)
    vaxseer_strain.columns = ["src_id1", "src_id2", "prediction", "label"]

    vaxseer_serum_H1N1 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/serum_holdout/h1n1/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_serum_H3N2 = pd.read_csv("~/workspace/vaxseer/runs/flu_hi_msa_regressor/serum_holdout/h3n2/max_steps_150k/TEST/lightning_logs/version_0/test_results.csv", index_col=False)
    vaxseer_serum = pd.concat([vaxseer_serum_H1N1, vaxseer_serum_H3N2], axis=0)
    vaxseer_serum.columns = ["src_id1", "src_id2", "prediction", "label"]

def abs_error_df(df, model):
    out = pd.DataFrame({
        "value": (df["label"].astype(float) - df["prediction"].astype(float)).abs(),
        "model": model,
    })
    return out.replace([np.inf, -np.inf], np.nan).dropna()

def concat_error_df(serumgate, nextflu, adaboost, vaxseer):
    return pd.concat([
        abs_error_df(serumgate, "SerumGate"),
        abs_error_df(nextflu, "Nextflu"),
        abs_error_df(adaboost, "Adaboost"),
        abs_error_df(vaxseer, "VaxSeer"),
    ], axis=0, ignore_index=True)

In [2]:


df_titer = concat_error_df(serumgate_titer, nextflu_titer, adaboost_titer, vaxseer_titer)
df_strain = concat_error_df(serumgate_strain, nextflu_strain, adaboost_strain, vaxseer_strain)
df_serum = concat_error_df(serumgate_serum, nextflu_serum, adaboost_serum, vaxseer_serum)

df_titer = df_titer[df_titer["value"] <= 4]
df_strain = df_strain[df_strain["value"] <= 4]
df_serum = df_serum[df_serum["value"] <= 8]

model_order = ["SerumGate", "Nextflu", "Adaboost", "VaxSeer"]
fixed_colors = ["#C5DFF4", "#F4EEAC", "#C9DCC4", "#F4B6C2"]

correlations = {
    "titer": [
        pearsonr(serumgate_titer["label"], serumgate_titer["prediction"])[0],
        pearsonr(nextflu_titer["label"], nextflu_titer["prediction"])[0],
        pearsonr(adaboost_titer["label"], adaboost_titer["prediction"])[0],
        pearsonr(vaxseer_titer["label"], vaxseer_titer["prediction"])[0],
    ],
    "strain": [
        pearsonr(serumgate_strain["label"], serumgate_strain["prediction"])[0],
        pearsonr(nextflu_strain["label"], nextflu_strain["prediction"])[0],
        pearsonr(adaboost_strain["label"], adaboost_strain["prediction"])[0],
        pearsonr(vaxseer_strain["label"], vaxseer_strain["prediction"])[0],
    ],
    "serum": [
        pearsonr(serumgate_serum["label"], serumgate_serum["prediction"])[0],
        pearsonr(nextflu_serum["label"], nextflu_serum["prediction"])[0],
        pearsonr(adaboost_serum["label"], adaboost_serum["prediction"])[0],
        pearsonr(vaxseer_serum["label"], vaxseer_serum["prediction"])[0],
    ],
}

sns.set(style="white")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

datasets = [
    (df_titer, axes[0], "divided by titer", "titer"),
    (df_strain, axes[1], "divided by strain", "strain"),
    (df_serum, axes[2], "divided by serum", "serum"),
]

for df, ax, title, key in datasets:
    sns.barplot(
        x="model",
        y="value",
        data=df,
        order=model_order,
        estimator=np.mean,
        errorbar="ci",
        capsize=0.2,
        ax=ax,
        palette=fixed_colors,
    )

    ax.set_title(title, fontsize=14)
    ax.set_ylabel("Titer Error", fontsize=14)
    ax.set_xlabel("")
    ax.grid(False)
    ax.tick_params(axis="both", which="major", labelsize=14)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    ax2 = ax.twinx()
    ax2.plot(range(len(model_order)), correlations[key], color="black", marker="o", linestyle="-")
    ax2.set_ylabel("Correlation", fontsize=14, color="black")
    ax2.tick_params(axis="y", labelsize=14, colors="black")
    ax2.set_ylim(-0.2, 1) if key == "serum" else ax2.set_ylim(0, 1)
    ax2.set_xticks(range(len(model_order)))

plt.tight_layout()
# plt.savefig("../Figure/Fig2BCD.svg", format="svg", transparent=True, bbox_inches="tight")
plt.show()

ValueError: cannot reindex on an axis with duplicate labels